In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os

# Get the current working directory
current_dir = os.getcwd()
print("Current Directory:", current_dir)
os.chdir(current_dir+"/../i24_data/")
# current_dir = os.getcwd()
# print("Current Directory:", current_dir)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# %%

# imports
import numpy as np
import torch
import _pickle as pickle
import pandas as pd
import matplotlib.pyplot as plt
import cv2
colors = np.random.rand(100000,3)

# import custom repo, use: pip3 install git+https://github.com/I24-MOTION/i24_rcs@v1.1-stable
from i24_rcs import I24_RCS  


# load each file

#reference images 
reference_dir = "./annotations/reference"

# homography
hg_cache_file = "./annotations/WACV2024_hg_save.cpkl"
hg_directory  = "./annotations/"                  

# rcs = I24_RCS(save_path = hg_cache_file, downsample = 2, default = "static")
# rcs.load_correspondences_WACV(hg_directory) 

# time-indexed data
detection_path = "./annotations/final_detections.npy"                    # each row is [time,x pos,y pos,length,width,height,veh class,detector confidence]
old_track_path = "./beta/MOTION_OLD_TIME.npy"     # each row is [time,x pos,y pos,length,width,height,veh class,id]
new_track_path = "./beta/MOTION_NEW_TIME.npy"     # each row is [time,x pos,y pos,length,width,height,veh class,id]
gps_data_path = "./annotations/final_gps.csv"                            # each row is [id,state plane x, state plane y,x,y,ts,length, width, height]


# trajectory-indexed data
old_trajectory_path = "./beta/old_0_3600_4000_20000.cpkl"     # dictionary indexed by id, each element is array with rows [time,x,y,l,w,h,class,ignore,ignore]]
new_trajectory_path = "./beta/new_0_3600_4000_20000.cpkl"     # dictionary indexed by id, each element is array with rows [time,x,y,l,w,h,class,ignore,ignore]]

path = new_trajectory_path # alias to old or new path


def load_MOTION_traj(file):
    try:
        with open(path,"rb") as f:
            tracklets,tracklets_terminated = pickle.load(f) 
    except:
        with open(path,"rb") as f:
            tracklets = pickle.load(f) 
            tracklets_terminated = []
           
    return tracklets + tracklets_terminated    

def traj_to_tensor(traj):
    t = torch.zeros(len(traj["x_position"]),6)
    t[:,0] = torch.tensor(traj["timestamp"])
    t[:,1] = torch.tensor(traj["x_position"])
    t[:,2] = torch.tensor(traj["y_position"])
    t[:,3] = traj["length"]
    t[:,4] = traj["width"]
    t[:,5] = traj["height"]
    return t

# tracklets = load_MOTION_traj(path)


##%% 9. Lets smooth a platoon
# if you reuse this code, please cite Yanbing Wang's Paper "Automatic vehicle trajectory data reconstruction at scale" from which the method is taken


from utils_opt import resample,opt2_l1_constr
lam1_x= 3e-1
lam2_x= 0
lam3_x= 1e-7

lam1_y= 0
lam2_y= 0
lam3_y= 1e-3



tracklets = load_MOTION_traj(path)


# you can select another set of ids manually by inspecting the above time-space plots
traj_ids = [46787,46614,46710,46796,1326,46981,46689,46653,674,46657,46728,46674,46616]
#traj_ids = [46787]



smoothed = []

# iterate over trajectories
for t,tidx in enumerate(traj_ids):
    print("On tracklet {}".format(tidx))
    traj = tracklets[tidx]
    
    # wrangle form for resampling
    car = {"timestamp" : traj[:,0].data.numpy(),
           "x_position": traj[:,1].data.numpy(),
           "y_position": traj[:,2].data.numpy(),
           "direction" : torch.sign(traj[0,2]).item(),
           "length":traj[0,3],
           "width" :traj[0,4],
           "height":traj[0,5]
           }
        
    re_car = resample(car,dt = 0.04,fillnan = True)
    smooth_car = opt2_l1_constr(re_car.copy(), lam2_x, lam2_y, lam3_x, lam3_y, lam1_x, lam1_y)
    re_car = traj_to_tensor(re_car)
    smooth_car = traj_to_tensor(smooth_car)
    smoothed.append([t,tidx,smooth_car])

##%% Plot the result

# define colors for plotting
c = np.linspace(0,1,num = len(traj_ids))
colors = np.zeros([len(traj_ids),3])
colors[:,0] = c
colors[:,2] = 1-c
#colors = np.random.rand(30,3)
    

fig, ax = plt.subplots(3,1, sharex=True, figsize=(10,8))
leg = []
for packet in smoothed:
    t,tidx,smooth_car = packet
    # plot position
    ax[0].plot(smooth_car[:,0],smooth_car[:,1],color = colors[t])
    
    
    # plot relative position relative to lead vehicle
    lead_idx = 0
    
    rel_x = []
    rel_xt = []
    for idx in range(smooth_car.shape[0]):
        while smooth_car[idx,0] < smoothed[0][2][0,0]:
            continue # skip all times before the lead car exists
            
        while lead_idx < smoothed[0][2].shape[0] and smoothed[0][2][lead_idx,0] < smooth_car[idx,0]:
            lead_idx += 1
        
        if lead_idx >= smoothed[0][2].shape[0]: break # skip all times after lead_car exists
        
        rel_x.append( smooth_car[idx,1] - smoothed[0][2][lead_idx,1])
        rel_xt.append(smooth_car[idx,0])
        
    ax[1].plot(rel_xt,torch.tensor(rel_x),color = colors[t])
            
    
    # plot speed
    ax[2].plot(smooth_car[:-1,0],-1*(smooth_car[1:,1]-smooth_car[:-1,1])/(smooth_car[1:,0]-smooth_car[:-1,0]),color = colors[t])
    leg.append("Platoon vehicle {} (id {})".format(t,tidx))


# Make plot pretty
ax[2].set_xlabel("Time (s)")
ax[1].set_ylabel("Distance Behind Lead Vehicle (ft)")
ax[0].set_ylabel("X Position (ft)")
ax[2].set_ylabel("Velocity (ft/s)")
fig.legend(leg)
ax[0].set_title("Platoon Plot")

# %% Extract more car-following pairs
# tracklets: [time,x,y,l,w,h,class, .., .., .., ..]

smoothed_all = []

# iterate over trajectories  # len(tracklets)
for t,tidx in enumerate(range(len(tracklets))):
    print("On tracklet {}".format(tidx))
    if tracklets[tidx].shape[0] < 100:
        print("Skip tracklet {}, it only has {} frames".format(tidx, tracklets[tidx].shape[0]))
        continue
    traj = tracklets[tidx]
    
    # wrangle form for resampling
    car = {"timestamp" : traj[:,0].data.numpy(),
           "x_position": traj[:,1].data.numpy(),
           "y_position": traj[:,2].data.numpy(),
           "direction" : torch.sign(traj[0,2]).item(),
           "length":traj[0,3],
           "width" :traj[0,4],
           "height":traj[0,5]
           }
        
    re_car = resample(car,dt = 0.04,fillnan = True)
    smooth_car = opt2_l1_constr(re_car.copy(), lam2_x, lam2_y, lam3_x, lam3_y, lam1_x, lam1_y)
    re_car = traj_to_tensor(re_car)
    smooth_car = traj_to_tensor(smooth_car)
    smoothed_all.append([t,tidx,smooth_car])
    
import pickle

# Open a file in binary write mode
with open('smoothed_all.pkl', 'wb') as f:
    # Serialize the dictionary and write it to the file
    pickle.dump(smoothed_all, f)